# 02 - Algoritmo Genético com Scorer no Threshold Operacional (0.40)

Este notebook replica os experimentos de `01_AG.ipynb` com uma diferença fundamental:
o scorer de recall e F2 avalia no **threshold 0.40** (operacional) em vez do 0.5 padrão do `predict()`.

**Por que isso importa?**  
No notebook 01_AG, o AG avaliava recall com `scoring="recall"` que usa `predict()` com threshold 0.5.
O modelo tem recall ~0.52 nesse threshold — impossível atingir o piso de 0.80.
Aqui usamos `make_scorer(..., needs_proba=True)` para avaliar no threshold 0.40,
omde o baseline já tem recall ~0.97. O piso de 0.80 é atingível → o AG otimiza F2 de verdade.

In [ ]:
import gc
import json
import math
import os
import pickle
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import Parallel, delayed

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    fbeta_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight
from utils.experiment_utils import apply_scenario

In [ ]:
SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

warnings.filterwarnings("ignore")

RANDOM_STATE          = 42
CV_SPLITS             = 3
MIN_RECALL_THRESHOLD  = 0.80
OPERATIONAL_THRESHOLD = 0.40
SCENARIO              = "B"

DATA_DIR      = Path("../data")
METRICS_DIR   = Path("../results/metrics")
ARTIFACTS_DIR = Path("../results/artifacts")
FIGURES_DIR   = Path("../results/figures")

def get_n_jobs(capacity: float = 0.80) -> int:
    """Retorna floor(n_cpus * capacity), mínimo 1."""
    return max(1, math.floor(os.cpu_count() * capacity))

# 01_AG.ipynb está rodando com 80% dos cores (9 de 12).
# Este notebook usa os que sobram, deixando 1 para o sistema.
_n_jobs_01ag = get_n_jobs(0.80)
N_JOBS = max(1, os.cpu_count() - _n_jobs_01ag - 1)
print(f"CPUs disponíveis : {os.cpu_count()}")
print(f"n_jobs (01_AG)   : {_n_jobs_01ag}")
print(f"n_jobs (02)     : {N_JOBS}  ← cores restantes − 1 para o sistema")

In [ ]:
def load_scenario_data(scenario: str) -> tuple:
    X_train_full = pd.read_parquet(DATA_DIR / "X_train.parquet")
    X_test_full  = pd.read_parquet(DATA_DIR / "X_test.parquet")
    y_train = pd.read_parquet(DATA_DIR / "y_train.parquet").iloc[:, 0]
    y_test  = pd.read_parquet(DATA_DIR / "y_test.parquet").iloc[:, 0]
    X_train = apply_scenario(X_train_full, "B")
    X_test  = apply_scenario(X_test_full,  "B")
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = load_scenario_data(SCENARIO)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Distribuição y_train:", y_train.value_counts(normalize=True).round(3).to_dict())

## Baseline (RandomizedSearch Fase 1)

In [ ]:
METRIC_NAMES = ["recall", "f2", "precision", "roc_auc", "average_precision", "brier"]

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold":          float(threshold),
        "recall":             recall_score(y_true, y_pred, zero_division=0),
        "f2":                 fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "f1":                 f1_score(y_true, y_pred, zero_division=0),
        "precision":          precision_score(y_true, y_pred, zero_division=0),
        "roc_auc":            roc_auc_score(y_true, y_prob),
        "average_precision":  average_precision_score(y_true, y_prob),
        "brier":              brier_score_loss(y_true, y_prob),
    }

def evaluate_model_on_test(params: dict, threshold: float = OPERATIONAL_THRESHOLD) -> dict:
    sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
    model = HistGradientBoostingClassifier(**params, random_state=RANDOM_STATE)
    model.fit(X_train, y_train, sample_weight=sample_weights)
    y_prob = model.predict_proba(X_test)[:, 1]
    return compute_metrics(y_test, y_prob, threshold=threshold)

def load_baseline_params(metrics_dir: Path) -> dict:
    operational = json.loads((metrics_dir / "best_model_operational_metrics.json").read_text())
    return operational["best_params"]

BASELINE_PARAMS = load_baseline_params(METRICS_DIR)

baseline_metrics_default     = evaluate_model_on_test(BASELINE_PARAMS, threshold=0.5)
baseline_metrics_operational = evaluate_model_on_test(BASELINE_PARAMS, threshold=OPERATIONAL_THRESHOLD)

baseline_summary = pd.DataFrame([
    {"threshold": "0.50 (padrão)",      **{m: baseline_metrics_default[m]     for m in METRIC_NAMES}},
    {"threshold": "0.40 (operacional)", **{m: baseline_metrics_operational[m] for m in METRIC_NAMES}},
]).set_index("threshold")

print("=== BASELINE — HistGB (RandomizedSearch Fase 1, sem calibração) ===")
display(baseline_summary.round(4))
print(f"\nMeta do AG: F2 > {baseline_metrics_operational['f2']:.4f} com recall >= {MIN_RECALL_THRESHOLD}")

gc.collect()

## Scorers customizados para threshold 0.40

O scorer padrão do sklearn (`scoring="recall"`) chama `predict()` que usa threshold 0.5.
Com `needs_proba=True`, o sklearn passa `predict_proba(X)[:, 1]` para a função de scoring,
permitindo aplicar manualmente o threshold 0.40.

Esta é a diferença central em relação ao `01_AG.ipynb`:
aqui o AG enxerga o threshold operacional durante o CV.
Recall >= 0.80 com threshold 0.40 é atingível pela maioria das configs → o AG otimiza F2 de verdade.

In [ ]:
FITNESS_PENALTY = -1.0

_cv_strategy = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Funções de scoring que avaliam no threshold operacional 0.40.
# **kwargs absorve argumentos extras que o sklearn pode passar internamente
# (ex: 'needs_proba' ou 'needs_threshold' em versões >= 1.4).
def _recall_at_operational(y_true, y_prob, **kwargs):
    y_pred = (y_prob >= OPERATIONAL_THRESHOLD).astype(int)
    return recall_score(y_true, y_pred, zero_division=0)

def _f2_at_operational(y_true, y_prob, **kwargs):
    y_pred = (y_prob >= OPERATIONAL_THRESHOLD).astype(int)
    return fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    
# needs_proba=True → sklearn passa predict_proba[:, 1] em vez de predict()
_recall_scorer_040 = make_scorer(_recall_at_operational, needs_proba=True)
_f2_scorer_040     = make_scorer(_f2_at_operational,     needs_proba=True)

def _cross_validate_recall_and_f2(model: HistGradientBoostingClassifier) -> tuple:
    sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
    fit_params = {"sample_weight": sample_weights}

    # n_jobs=1: paralelismo feito em _evaluate_population via joblib.Parallel
    mean_recall = cross_val_score(
        model, X_train, y_train,
        cv=_cv_strategy,
        scoring=_recall_scorer_040,
        params=fit_params,
        n_jobs=1,
    ).mean()

    mean_f2 = cross_val_score(
        model, X_train, y_train,
        cv=_cv_strategy,
        scoring=_f2_scorer_040,
        params=fit_params,
        n_jobs=1,
    ).mean()

    return float(mean_recall), float(mean_f2)

## Representação genética

Idêntica ao `01_AG.ipynb` — mesmo espaço de busca (6.300 combinações), mesma codificação por índice.

In [ ]:
SEARCH_SPACE: dict = {
    "max_iter":          [100, 150, 200, 250, 300, 350, 400],
    "learning_rate":     [0.01, 0.03, 0.05, 0.08, 0.1, 0.15],
    "max_leaf_nodes":    [15, 23, 31, 47, 63],
    "min_samples_leaf":  [20, 35, 50, 75, 100],
    "l2_regularization": [0.0, 0.01, 0.1, 0.5, 1.0, 2.0],
}

PARAM_NAMES  = list(SEARCH_SPACE.keys())
PARAM_VALUES = list(SEARCH_SPACE.values())
N_GENES      = len(PARAM_NAMES)

def decode(individual: np.ndarray) -> dict:
    return {PARAM_NAMES[i]: PARAM_VALUES[i][individual[i]] for i in range(N_GENES)}

def random_individual(rng: np.random.Generator) -> np.ndarray:
    return np.array([rng.integers(0, len(values)) for values in PARAM_VALUES])

# Sanidade
_rng    = np.random.default_rng(RANDOM_STATE)
_sample = random_individual(_rng)
print("Cromossomo :", _sample)
print("Parâmetros :", decode(_sample))

## Função fitness (threshold 0.40)

Mesma lógica do `01_AG.ipynb`, mas agora `_cross_validate_recall_and_f2` usa os scorers de threshold 0.40.
Com isso, o piso de 0.80 é atingível e o AG entra direto na zona de F2.

In [ ]:
def fitness(individual: np.ndarray, min_recall_threshold: float = MIN_RECALL_THRESHOLD) -> float:
    model = HistGradientBoostingClassifier(**decode(individual), random_state=RANDOM_STATE)
    mean_recall, mean_f2 = _cross_validate_recall_and_f2(model)
    meets_recall_floor = mean_recall >= min_recall_threshold
    return mean_f2 if meets_recall_floor else mean_recall + FITNESS_PENALTY

In [ ]:
# Verificação: baseline deve passar o piso de 0.80 com threshold 0.40
_baseline_individual = np.array([5, 2, 2, 4, 4])
print("Parâmetros baseline:", decode(_baseline_individual))

_baseline_cv_recall, _baseline_cv_f2 = _cross_validate_recall_and_f2(
    HistGradientBoostingClassifier(**BASELINE_PARAMS, random_state=RANDOM_STATE)
)
print(f"\nCV com scorer threshold 0.40:")
print(f"  Recall médio : {_baseline_cv_recall:.4f}  (piso 0.80: {'PASSA ✓' if _baseline_cv_recall >= 0.80 else 'FALHA ✗'})")
print(f"  F2 médio     : {_baseline_cv_f2:.4f}")
print(f"\nFitness do baseline: {fitness(_baseline_individual):.4f}")
print("(fitness positivo = AG opera em F2 desde a primeira geração)")

## Operadores genéticos

Idênticos ao `01_AG.ipynb` — seleção por torneio, crossover de ponto único, mutação por gene.

In [ ]:
def tournament_selection(population, fitnesses, tournament_size, rng):
    candidates    = rng.choice(len(population), size=tournament_size, replace=False)
    winner_index  = candidates[np.argmax([fitnesses[i] for i in candidates])]
    return population[winner_index].copy()

def single_point_crossover(parent_a, parent_b, crossover_rate, rng):
    if rng.random() > crossover_rate:
        return parent_a.copy(), parent_b.copy()
    cut_point = rng.integers(1, N_GENES)
    child_a = np.concatenate([parent_a[:cut_point], parent_b[cut_point:]])
    child_b = np.concatenate([parent_b[:cut_point], parent_a[cut_point:]])
    return child_a, child_b

def mutate(individual, mutation_rate, rng):
    mutated = individual.copy()
    for gene_index in range(N_GENES):
        if rng.random() < mutation_rate:
            mutated[gene_index] = rng.integers(0, len(PARAM_VALUES[gene_index]))
    return mutated

## Loop principal do algoritmo genético

In [ ]:
def _initialize_population(pop_size: int, rng: np.random.Generator) -> list:
    return [random_individual(rng) for _ in range(pop_size)]

def _evaluate_population(population: list, min_recall_threshold: float = MIN_RECALL_THRESHOLD) -> list:
    # N_JOBS indivíduos avaliados em paralelo; CV de cada um roda sequencialmente (n_jobs=1)
    return Parallel(n_jobs=N_JOBS)(
        delayed(fitness)(individual, min_recall_threshold) for individual in population
    )

def _breed_next_generation(
    population, fitnesses, pop_size, elite_size,
    tournament_size, crossover_rate, mutation_rate, rng,
):
    sorted_by_fitness = np.argsort(fitnesses)[::-1]
    next_generation   = [population[i].copy() for i in sorted_by_fitness[:elite_size]]

    while len(next_generation) < pop_size:
        parent_a = tournament_selection(population, fitnesses, tournament_size, rng)
        parent_b = tournament_selection(population, fitnesses, tournament_size, rng)
        child_a, child_b = single_point_crossover(parent_a, parent_b, crossover_rate, rng)
        next_generation.append(mutate(child_a, mutation_rate, rng))
        if len(next_generation) < pop_size:
            next_generation.append(mutate(child_b, mutation_rate, rng))

    return next_generation

def run_genetic_algorithm(
    pop_size: int               = 20,
    n_generations: int          = 10,
    mutation_rate: float        = 0.20,
    crossover_rate: float       = 0.80,
    tournament_size: int        = 3,
    elite_size: int             = 1,
    seed: int                   = RANDOM_STATE,
    label: str                  = "AG",
    min_recall_threshold: float = MIN_RECALL_THRESHOLD,
) -> dict:
    rng        = np.random.default_rng(seed)
    population = _initialize_population(pop_size, rng)

    best_individual   = None
    best_fitness_ever = -np.inf
    history_rows      = []

    for generation in range(1, n_generations + 1):
        fitnesses = _evaluate_population(population, min_recall_threshold)

        generation_best_index   = int(np.argmax(fitnesses))
        generation_best_fitness = fitnesses[generation_best_index]

        if generation_best_fitness > best_fitness_ever:
            best_fitness_ever = generation_best_fitness
            best_individual   = population[generation_best_index].copy()

        history_rows.append({
            "generation":   generation,
            "best_fitness": generation_best_fitness,
            "mean_fitness": float(np.mean(fitnesses)),
            "best_params":  decode(population[generation_best_index]),
        })

        print(
            f"[{label}] Geração {generation}/{n_generations} | "
            f"melhor={generation_best_fitness:.4f} | "
            f"média={np.mean(fitnesses):.4f}"
        )

        population = _breed_next_generation(
            population, fitnesses, pop_size, elite_size,
            tournament_size, crossover_rate, mutation_rate, rng,
        )

    return {
        "label":                label,
        "best_individual":      best_individual,
        "best_params":          decode(best_individual),
        "best_cv_fitness":      best_fitness_ever,
        "min_recall_threshold": min_recall_threshold,
        "history":              pd.DataFrame(history_rows),
    }

In [ ]:
# Smoke test — deve terminar com fitness positivo (AG já em F2)
_smoke = run_genetic_algorithm(
    pop_size=4, n_generations=2, label="smoke_040", seed=RANDOM_STATE
)

print("\nSmoke test OK")
print("Params  :", _smoke["best_params"])
print(f"Fitness : {_smoke['best_cv_fitness']:.4f}")
print("(fitness positivo confirma que o AG opera em F2 com threshold 0.40)")

assert all(
    v in PARAM_VALUES[i]
    for i, v in enumerate(_smoke["best_params"].values())
), "Algum valor fora do espaço de busca!"
print("Validação do espaço: OK")

## Experimentos — scorer threshold 0.40 | recall >= 0.80

Com o scorer no threshold 0.40, o piso de 0.80 é atingível pela maioria das configs.
Usamos um único cenário de recall (0.80 — o piso clínico original) porque não há
necessidade de documentar pisos alternativos: o AG já funciona corretamente.

Os três primeiros experimentos replicam as configurações do `01_AG.ipynb` para comparação direta.
O quarto testa **inicialização com hot start**: a população começa com o melhor indivíduo
da Fase 1 (RandomizedSearch) e o restante aleatório — avalia se partir de uma solução
conhecida boa acelera a convergência ou encontra algo melhor ao redor desse ponto.

| Experimento | pop_size | n_generations | mutation_rate | crossover_rate | Perfil |
|---|---|---|---|---|---|
| Exp 1 | 10 | 8 | 0.10 | 0.70 | Conservador |
| Exp 2 | 20 | 15 | 0.20 | 0.80 | Padrão |
| Exp 3 | 20 | 15 | 0.40 | 0.80 | Exploratório |
| Exp 4 | 20 | 15 | 0.20 | 0.80 | Hot start (seed = baseline Fase 1) |


In [ ]:
# Scorer correto: piso 0.80 é atingível com threshold 0.40 → cenário único
RECALL_SCENARIOS = [0.80]

EXPERIMENT_CONFIGS = [
    dict(pop_size=10, n_generations=8,  mutation_rate=0.10, crossover_rate=0.70, label="Exp1_conservador"),
    dict(pop_size=20, n_generations=15, mutation_rate=0.20, crossover_rate=0.80, label="Exp2_padrao"),
    dict(pop_size=20, n_generations=15, mutation_rate=0.40, crossover_rate=0.80, label="Exp3_exploratorio"),
]

In [ ]:
scenario_results: dict = {}  # {recall_threshold: {label: result}}

for recall_threshold in RECALL_SCENARIOS:
    print(f"\n{'#'*60}")
    print(f"# CENÁRIO: recall >= {recall_threshold} | scorer threshold {OPERATIONAL_THRESHOLD}")
    print(f"{'#'*60}")

    experiment_results: dict = {}

    for config in EXPERIMENT_CONFIGS:
        print(f"\n{'='*60}\nIniciando {config['label']}\n{'='*60}")

        result = run_genetic_algorithm(
            **config,
            min_recall_threshold=recall_threshold,
            seed=RANDOM_STATE,
        )
        experiment_results[config["label"]] = result

        print(f"Melhores parâmetros: {result['best_params']}")
        print(f"Melhor fitness CV:   {result['best_cv_fitness']:.4f}")

    scenario_results[recall_threshold] = experiment_results

In [ ]:
def _initialize_population_seeded(pop_size, rng, seed_individual):
    """Inicializa com baseline da Fase 1 como primeiro indivíduo."""
    pop = [seed_individual.copy()]
    pop += [random_individual(rng) for _ in range(pop_size - 1)]
    return pop

def run_genetic_algorithm_seeded(
    seed_individual,
    pop_size=20, n_generations=15, mutation_rate=0.20, crossover_rate=0.80,
    tournament_size=3, elite_size=1, seed=RANDOM_STATE, label="AG",
    min_recall_threshold=MIN_RECALL_THRESHOLD,
):
    rng        = np.random.default_rng(seed)
    population = _initialize_population_seeded(pop_size, rng, seed_individual)

    best_individual   = None
    best_fitness_ever = -np.inf
    history_rows      = []

    for generation in range(1, n_generations + 1):
        fitnesses = _evaluate_population(population, min_recall_threshold)

        generation_best_index   = int(np.argmax(fitnesses))
        generation_best_fitness = fitnesses[generation_best_index]

        if generation_best_fitness > best_fitness_ever:
            best_fitness_ever = generation_best_fitness
            best_individual   = population[generation_best_index].copy()

        history_rows.append({
            "generation":   generation,
            "best_fitness": generation_best_fitness,
            "mean_fitness": float(np.mean(fitnesses)),
            "best_params":  decode(population[generation_best_index]),
        })

        print(
            f"[{label}] Geração {generation}/{n_generations} | "
            f"melhor={generation_best_fitness:.4f} | "
            f"média={np.mean(fitnesses):.4f}"
        )

        population = _breed_next_generation(
            population, fitnesses, pop_size, elite_size,
            tournament_size, crossover_rate, mutation_rate, rng,
        )

    return {
        "label":                label,
        "best_individual":      best_individual,
        "best_params":          decode(best_individual),
        "best_cv_fitness":      best_fitness_ever,
        "min_recall_threshold": min_recall_threshold,
        "history":              pd.DataFrame(history_rows),
    }

In [ ]:
# _baseline_individual = np.array([5, 2, 2, 4, 4]) — já definido na célula 13
result_exp4 = run_genetic_algorithm_seeded(
    seed_individual=_baseline_individual,
    pop_size=20, n_generations=15,
    mutation_rate=0.20, crossover_rate=0.80,
    label="Exp4_hotstart",
)

print("\nExp4 concluído")
print("Params  :", result_exp4["best_params"])
print(f"Fitness : {result_exp4['best_cv_fitness']:.4f}")
scenario_results[0.80]["Exp4_hotstart"] = result_exp4

In [ ]:
for recall_threshold, experiment_results in scenario_results.items():
    fig, axes = plt.subplots(1, len(experiment_results), figsize=(20, 4), sharey=True)

    for ax, (label, result) in zip(axes, experiment_results.items()):
        history = result["history"]
        ax.plot(history["generation"], history["best_fitness"], marker="o", label="melhor geração")
        ax.plot(history["generation"], history["mean_fitness"], linestyle="--", label="média da população")
        ax.axhline(
            baseline_metrics_operational["f2"],
            color="red", linestyle=":", linewidth=1.5, label="baseline F2",
        )
        ax.set_title(label)
        ax.set_xlabel("Geração")
        ax.set_ylabel("Fitness F2 (threshold 0.40)")
        ax.legend(fontsize=8)

    plt.suptitle(
        f"Convergência — scorer threshold 0.40 | recall >= {recall_threshold}",
        fontsize=13,
    )
    plt.tight_layout()
    filename = f"02_ag_convergence_040_recall{str(recall_threshold).replace('.', '')}.png"
    plt.savefig(FIGURES_DIR / filename, dpi=150)
    plt.show()

## Comparativo final: baseline vs AG (scorer threshold 0.40)

Avaliação no conjunto de teste com threshold 0.40 — mesma metodologia do `01_AG.ipynb`.
O resultado deste notebook é a resposta à pergunta: **o AG melhora o modelo quando enxerga o threshold correto?**

In [ ]:
def build_comparison_row(label: str, params: dict, cv_fitness: float) -> dict:
    metrics = evaluate_model_on_test(params, threshold=OPERATIONAL_THRESHOLD)
    return {
        "origem":     label,
        "params":     str(params),
        "cv_fitness": round(cv_fitness, 4),
        **{m: metrics[m] for m in METRIC_NAMES},
    }

scenario_comparison_dfs: dict = {}

for recall_threshold, experiment_results in scenario_results.items():
    comparison_rows = [
        build_comparison_row(
            label="Baseline (RandomizedSearch Fase 1)",
            params=BASELINE_PARAMS,
            cv_fitness=fitness(_baseline_individual, recall_threshold),
        ),
        *[
            build_comparison_row(
                label=label,
                params=result["best_params"],
                cv_fitness=result["best_cv_fitness"],
            )
            for label, result in experiment_results.items()
        ]
    ]

    df = pd.DataFrame(comparison_rows).set_index("origem")
    scenario_comparison_dfs[recall_threshold] = df

    print(f"\n=== Scorer threshold 0.40 | recall >= {recall_threshold} no CV ===")
    display(df.round(4))

    filename = f"02_ag_comparison_040_recall{str(recall_threshold).replace('.', '')}.csv"
    df.to_csv(METRICS_DIR / filename)
    print(f"Salvo: {METRICS_DIR / filename}")

gc.collect()

In [ ]:
PIVOT_THRESHOLD_FAIL    = 0.01
PIVOT_THRESHOLD_SUCCESS = 0.03
MAX_FP_GROWTH_PCT       = 0.15
MAX_FN_GROWTH_PCT       = 0.30
MIN_RECALL_TEST         = 0.80

for recall_threshold, comparison_df in scenario_comparison_dfs.items():
    print(f"\n{'='*60}")
    print(f"=== Conclusão — scorer 0.40 | recall >= {recall_threshold} no CV ===")
    print(f"{'='*60}")

    best_experiment = (
        comparison_df
        .drop("Baseline (RandomizedSearch Fase 1)")
        .sort_values("f2", ascending=False)
        .iloc[0]
    )
    baseline_row = comparison_df.loc["Baseline (RandomizedSearch Fase 1)"]

    delta_f2        = best_experiment["f2"]        - baseline_row["f2"]
    delta_recall    = best_experiment["recall"]    - baseline_row["recall"]
    delta_precision = best_experiment["precision"] - baseline_row["precision"]

    print(f"Melhor experimento AG : {best_experiment.name}")
    print(f"F2       : {best_experiment['f2']:.4f}   (baseline: {baseline_row['f2']:.4f}   | delta: {delta_f2:+.4f})")
    print(f"Recall   : {best_experiment['recall']:.4f}   (baseline: {baseline_row['recall']:.4f}   | delta: {delta_recall:+.4f})")
    print(f"Precisão : {best_experiment['precision']:.4f}   (baseline: {baseline_row['precision']:.4f}   | delta: {delta_precision:+.4f})")
    print(f"ROC-AUC  : {best_experiment['roc_auc']:.4f}   (baseline: {baseline_row['roc_auc']:.4f})")

    n_prematuros_test = int(y_test.sum())
    fn_baseline = int(n_prematuros_test * (1 - baseline_row["recall"]))
    fn_ag       = int(n_prematuros_test * (1 - best_experiment["recall"]))
    tp_baseline = n_prematuros_test - fn_baseline
    tp_ag       = n_prematuros_test - fn_ag
    fp_baseline = int(tp_baseline / max(baseline_row["precision"], 1e-9) - tp_baseline)
    fp_ag       = int(tp_ag       / max(best_experiment["precision"], 1e-9) - tp_ag)
    fp_growth   = (fp_ag - fp_baseline) / max(fp_baseline, 1)
    fn_growth   = (fn_ag - fn_baseline) / max(fn_baseline, 1)

    print(f"\nEstimativa no teste (threshold={OPERATIONAL_THRESHOLD}):")
    print(f"{'':30} {'Baseline':>10} {'Melhor AG':>10} {'Delta':>10} {'Crescimento':>12}")
    print(f"{'FN (prematuros perdidos)':30} {fn_baseline:>10,} {fn_ag:>10,} {fn_ag - fn_baseline:>+10,} {fn_growth:>+11.1%}")
    print(f"{'FP (alarmes falsos)':30} {fp_baseline:>10,} {fp_ag:>10,} {fp_ag - fp_baseline:>+10,} {fp_growth:>+11.1%}")

    recall_ok    = best_experiment["recall"] >= MIN_RECALL_TEST
    fn_ok        = fn_growth <= MAX_FN_GROWTH_PCT
    fp_ok        = fp_growth <= MAX_FP_GROWTH_PCT
    validacao_ok = recall_ok and fn_ok and fp_ok

    print("\n--- Validação clínica ---")
    print(f"  Recall >= {MIN_RECALL_TEST}       : {'✓' if recall_ok else '✗'}  {best_experiment['recall']:.4f}")
    print(f"  FN cresce <= {MAX_FN_GROWTH_PCT:.0%}    : {'✓' if fn_ok else '✗'}  {fn_growth:+.1%}  ({fn_ag - fn_baseline:+,} prematuros)")
    print(f"  FP cresce <= {MAX_FP_GROWTH_PCT:.0%}    : {'✓' if fp_ok else '✗'}  {fp_growth:+.1%}  ({fp_ag - fp_baseline:+,} alarmes)")

    if not recall_ok:
        print("  ⚠ Recall abaixo do piso clínico")
    if not fn_ok:
        print(f"  ⚠ FN cresceu {fn_growth:+.1%} — {fn_ag - fn_baseline:+,} prematuros não identificados a mais")
    if not fp_ok:
        print(f"  ⚠ FP cresceu {fp_growth:+.1%} — alarmes em nível inaceitável")

    print("\n--- Critério de pivô ---")
    if not validacao_ok:
        print("✗ Falha na validação clínica — F2 elevado não representa melhora clínica real")
    elif delta_f2 >= PIVOT_THRESHOLD_SUCCESS:
        print(f"✓ Melhoria expressiva ({delta_f2:+.4f}) — AG com scorer correto bem-sucedido.")
        print("  Recomendação: seguir com os parâmetros encontrados pelo AG.")
    elif delta_f2 >= PIVOT_THRESHOLD_FAIL:
        print(f"~ Melhoria marginal ({delta_f2:+.4f}) — combinar com threshold estratificado.")
        print("  Recomendação: aplicar threshold estratificado por perfil de risco.")
    else:
        print(f"✗ Sem melhoria relevante ({delta_f2:+.4f}) — dados limitam o modelo.")
        print("  Recomendação: threshold estratificado ou feature engineering.")